In [ ]:
%matplotlib inline

import os
import shutil
from glob import glob
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from loguru import logger
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
# GLOBAL PARAMS
images_csv = 'wildlife-insights_4fc92e40-94fe-48d3-a51f-5ac1a03f5e03_project-2007160_data/images_2007160.csv'
deployments_csv = 'wildlife-insights_4fc92e40-94fe-48d3-a51f-5ac1a03f5e03_project-2007160_data/deployments.csv'
pickup_date = '2024-03-23'
dataset_name = 'pilot_mnt_data'
device = 'cuda'
cls_model_weights = 'yolov8x.pt'

In [ ]:
def change_file_time(file, t):
    timestamp_str = str(t)
    parsed_time = time.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
    ts = time.mktime(parsed_time)
    os.utime(file, (ts, ts))

In [ ]:
def get_mask_image(im_path, display_images=False):
    results = model.predict(source=image_path, classes=[0])
    boxes = results[0].boxes.xyxy.tolist()
    if not boxes:
        logger.error(f'Could not find a human in {image_path}!')
        return image_path
    if len(boxes) > 1:
        bbox1_area = calculate_area(boxes[0])
        bbox2_area = calculate_area(boxes[1])
        if bbox1_area > bbox2_area:
            idx = 1
        elif bbox2_area > bbox1_area:
            idx = 0
    else:
        idx = 0

    if image_path in switch_index:
        idx = idx + 1 if idx == 0 else idx - 1
        if len(boxes) != 2:
            logger.error(f'Could not find calib-human in {image_path}!')
            return image_path

    bbox = boxes[idx]

    image = cv2.cvtColor(cv2.imread(im_path), cv2.COLOR_BGR2RGB)
    predictor.set_image(image)

    input_box = np.array(bbox)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box,
        multimask_output=False,
    )

    segmentation_mask = masks[0]

    # Convert the segmentation mask to a binary mask
    binary_mask = np.where(segmentation_mask > 0.5, 1, 0)
    visualize_contours(image, binary_mask)

    # Create white background and black background
    white_background = np.ones_like(image) * 255
    black_background = np.zeros_like(image)

    # Apply the binary mask to the white background
    new_image = white_background * binary_mask[..., np.newaxis] + black_background * (1 - binary_mask[..., np.newaxis])

    plt.imsave(f'{deployment_path}/calibration_frames_masks/{Path(im_path).name}', new_image.astype(np.uint8))
    if display_images:
        plt.imshow(new_image.astype(np.uint8))
        plt.axis('off')
        plt.show()

In [ ]:
df = pd.read_csv(images_csv)
print('df length:', len(df))
print('Number of deployments in df:', len(df['deployment_id'].unique()))

df_deployments = pd.read_csv('wildlife-insights_4fc92e40-94fe-48d3-a51f-5ac1a03f5e03_project-2007160_data/deployments.csv')
df = df.merge(df_deployments[['camera_name', 'deployment_id']], on='deployment_id', how='inner')

df['deployment_id'] = df['deployment_id'].str.replace('/', '-')
df['timestamp'] = pd.to_datetime(df['timestamp'])

df_animals = df[~df['common_name'].isin(['Human-Camera Trapper', 'Human-Faces', 'Human', 'Blank'])]
print('Animals df length:', len(df_animals))

df_animals = df_animals.sort_values(by='timestamp')
df_animals.reset_index(drop=True, inplace=True)

df_animals['location'].to_csv('animal_images.csv', index=False, header=False)
Path('animal_images').mkdir(exist_ok=True)

! cat 'images_animals.csv' | gsutil -m cp -I 'animal_images'

for x in df_animals['deployment_id'].unique():
    Path(f'animal_images/{x}').mkdir(exist_ok=True, parents=True)

for x, y, z, t in zip(df_animals['location'], df_animals['deployment_id'], df_animals['image_id'], df_animals['timestamp']):
    out_path = f'animal_images/{y}/{z}{Path(x).suffix}'
    shutil.move(f'animal_images/{Path(x).name}', out_path)
    change_file_time(out_path, t)


In [ ]:
print(f'Cameras pick-up date: {pickup_date}')
df_human = df[df['timestamp'].dt.date == pd.to_datetime(pickup_date).date()]

df_human = df_human[df_human['common_name'].isin(['Human-Faces'])]
print('Animals df length:', len(df_human))

df_human['location'].to_csv('human_images.csv', index=False, header=False)
Path('human_images').mkdir(exist_ok=True)

! cat 'human_images.csv' | gsutil -m cp -I 'human_images'

for x in df_human['deployment_id'].unique():
    Path(f'human_images/{x}').mkdir(exist_ok=True, parents=True)

for x, y, z, t in zip(df_human['location'], df_human['deployment_id'], df_human['image_id'], df_human['timestamp']):
    out_path = f'human_images/{y}/{z}{Path(x).suffix}'
    shutil.move(f'human_images/{Path(x).name}', out_path)
    change_file_time(out_path, t)


In [ ]:
human = glob('human_images/[!_]*')
animal = glob('animal_images/[!_]*')

h = [Path(x).name for x in human]
a = [Path(x).name for x in animal]
assert not [x for x in a if x not in h]

#---------------------------------------------------
# WARNING !
# SPECIFIC TO ONE DATASET, DON'T USE IN PRODUCTION
mapping = {
    'HC500 Hyperfire Semi-Covert IR': 'Reconyx HC500 Hyperfire',
    'HC500 Hyperfire Semi-Covert IR_': 'Reconyx HC500 Hyperfire'
}
df['camera_name'] = df['camera_name'].replace(mapping)
# WARNING !
#---------------------------------------------------

for camera_name in df['camera_name'].unique():
    Path(f'{dataset_name}/{camera_name}/transects').mkdir(exist_ok=True, parents=True)
    Path(f'{dataset_name}/{camera_name}/results').mkdir(exist_ok=True)

    for deployment in h:
        for subfolder in ['calibration_frames', 'calibration_frames_masks', 'detection_frames']:
            Path(f'{dataset_name}/{camera_name}/transects/{deployment}/{subfolder}').mkdir(exist_ok=True, parents=True)

    for deployment in human:
        for image in glob(f'{deployment}/*'):
            out_path = f'{dataset_name}/{camera_name}/transects/{Path(deployment).name}/calibration_frames'
            shutil.copy2(image, out_path)

    for deployment in animal:
        for image in glob(f'{deployment}/*'):
            out_path = f'{dataset_name}/{camera_name}/transects/{Path(deployment).name}/detection_frames'
            shutil.copy2(image, out_path)


In [ ]:
transects = glob(f'{dataset_name}/*/transects/*')

model = YOLO(cls_model_weights)

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)

failed_images = []

for deployment_path in transects:
    ims = glob(f'{deployment_path}/calibration_frames/*')
    ims.sort()
    for im_path in ims:
        failed_image_path = get_mask_image(im_path, display_images=False)
        if failed_image_path:
            logger.warning(f'Failed to extract mask from: {failed_image_path}')
            Path(im_path).unlink()


In [ ]:
# RUN DEPTH ESTIMATION MODEL...

In [ ]:
for camera_name in df['camera_name'].unique():
    results_df = pd.read_csv(f'{dataset_name}/{camera_name}/results/results.csv')
    deployment_ids = df['deployment_id'].unique()
    results_df['image_id'] = results_df['frame_id']

    results_df = pd.merge(df, results_df, on='image_id', how='inner')
    results_df.to_csv(f'{dataset_name}/{camera_name}/results/results-{camera_name}.csv', index=False)
